# Food Orders 

### Business scenario
The operations team of an online food-delivery company has shared order-level information covering customers, restaurants, payments, delivery, refunds, weather and order channels. Before this information is used for reporting, I need to understand the dataset, identify data-quality issues, decide suitable cleaning rules, clean the data and create useful analytical columns.

### Objectives
1. Understand the structure and meaning of the dataset.
2. Assess missing values, duplicates, data types and invalid records.
3. Decide and document cleaning rules before changing the data.
4. Clean the data while preserving the original dataset.
5. Create and validate derived columns for delivery and platform revenue.

### Working approach
- `raw_df` stores the original CSV data and will not be modified.
- `work_df` is a separate copy used for cleaning and transformation.
- Missing financial values will not automatically be replaced with zero or an average.
- Missing payment information will be labelled `Unknown`.
- Missing offer information will be labelled `Not Specified`; this does not mean that no offer was applied.
- Suspicious values will be flagged for investigation rather than silently deleted.
- Keep `food_orders.csv` in the same folder as this notebook and run all cells from top to bottom.

## Setup — Import libraries and load the CSV

I use **Pandas** to read and analyse tabular data and **NumPy** for numerical operations and missing-value handling. The CSV is loaded into `raw_df`, which is kept unchanged throughout the notebook.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

raw_df = pd.read_csv("food_orders.csv")

print("Dataset loaded successfully.")
print("Original dataset shape:", raw_df.shape)

Dataset loaded successfully.
Original dataset shape: (1010, 14)


# Task 1 — Load and understand the dataset

The purpose of this task is to understand what information is available before making any changes. I first inspect sample records, the dataset dimensions, column names, data types and summary statistics.

### 1.1 View the first five, last five and five random records

These views help me see the beginning, end and a small selection of records from different parts of the dataset. A fixed `random_state` makes the sample reproducible.

In [2]:
print("First five records")
display(raw_df.head())

print("Last five records")
display(raw_df.tail())

print("Five random records")
display(raw_df.sample(n=min(5, len(raw_df)), random_state=42))

First five records


,Order ID,Customer ID,Restaurant ID,Order Date and Time,Delivery Date and Time,Order Value,Delivery Fee,Payment Method,Discounts and Offers,Commission Fee,Payment Processing Fee,Refunds/Chargebacks,Weather Condition,Order Channel
0,1,C8270,R2924,2024-02-01 01:11:52,2024-02-01 02:39:52,1914.0,0.0,Credit Card,5% on App,150.0,47.0,0.0,Rainy,Mobile App
1,2,C1860,R2054,2024-02-02 22:11:04,2024-02-02 22:46:04,986.0,40.0,Digital Wallet,10%,198.0,23.0,0.0,Cloudy,Mobile App
2,3,C6390,R2870,2024-01-31 05:54:35,2024-01-31 06:52:35,937.0,30.0,Cash on Delivery,15% New User,195.0,45.0,0.0,Humid,Mobile App
3,4,C6191,R2642,2024-01-16 22:52:49,2024-01-16 23:38:49,1463.0,NaN,Cash on Delivery,NaN,146.0,27.0,0.0,Rainy,Mobile App
4,5,C6734,R2799,2024-01-29 01:19:30,2024-01-29 02:48:30,1992.0,30.0,Cash on Delivery,50 off Promo,130.0,50.0,NaN,Sunny,Mobile App


Last five records


,Order ID,Customer ID,Restaurant ID,Order Date and Time,Delivery Date and Time,Order Value,Delivery Fee,Payment Method,Discounts and Offers,Commission Fee,Payment Processing Fee,Refunds/Chargebacks,Weather Condition,Order Channel
1005,208,C5736,R2750,2024-01-18 09:08:01,2024-01-18 09:49:01,NaN,50.0,Digital Wallet,50 off Promo,184.0,41.0,0.0,Rainy,Mobile App
1006,580,C8102,R2257,2024-01-03 08:46:06,2024-01-03 10:10:06,1045.0,40.0,Cash on Delivery,10%,137.0,13.0,0.0,Sunny,Partner App
1007,497,C9010,R2462,2024-02-04 13:07:32,2024-02-04 14:25:32,1125.0,0.0,Credit Card,10%,95.0,16.0,50.0,Cloudy,Partner App
1008,197,C2986,R2097,2024-01-16 13:11:39,2024-01-16 14:04:39,841.0,0.0,Cash on Delivery,15% New User,105.0,42.0,50.0,Cloudy,Mobile App
1009,133,C6915,R2074,2024-01-20 09:37:53,2024-01-20 11:05:53,1137.0,40.0,Credit Card,NaN,50.0,50.0,150.0,Cloudy,Mobile App


Five random records


,Order ID,Customer ID,Restaurant ID,Order Date and Time,Delivery Date and Time,Order Value,Delivery Fee,Payment Method,Discounts and Offers,Commission Fee,Payment Processing Fee,Refunds/Chargebacks,Weather Condition,Order Channel
629,630,C1716,R2905,2024-01-13 04:16:47,2024-01-13 06:09:47,NaN,0.0,Cash on Delivery,5% on App,91.0,42.0,100.0,Rainy,Mobile App
788,789,C9659,R2778,2024-01-31 10:35:46,2024-01-31 12:07:46,714.0,40.0,Cash on Delivery,NaN,178.0,25.0,150.0,Rainy,Partner App
684,685,C7254,R2538,2024-01-07 23:31:43,2024-01-08 01:04:43,204.0,40.0,Digital Wallet,NaN,195.0,11.0,0.0,Cloudy,Mobile App
516,517,C8494,R2687,2024-01-23 05:41:17,2024-01-23 06:24:17,948.0,50.0,NaN,NaN,106.0,28.0,0.0,Sunny,Mobile App
529,530,C5268,R2697,2024-02-04 18:17:27,2024-02-04 18:51:27,1815.0,50.0,Cash on Delivery,15% New User,114.0,33.0,0.0,Sunny,Mobile App


### 1.2 Check the number of rows, columns and column names

The shape tells me how many records and fields are available. Column names show which business details have been collected.

In [3]:
print("Number of rows:", raw_df.shape[0])
print("Number of columns:", raw_df.shape[1])

print("\nColumn names:")
for column in raw_df.columns:
    print("-", column)

Number of rows: 1010
Number of columns: 14

Column names:
- Order ID
- Customer ID
- Restaurant ID
- Order Date and Time
- Delivery Date and Time
- Order Value
- Delivery Fee
- Payment Method
- Discounts and Offers
- Commission Fee
- Payment Processing Fee
- Refunds/Chargebacks
- Weather Condition
- Order Channel


### 1.3 Check the current data types

`info()` shows each column's data type and non-null count. This helps identify fields, such as timestamps, that may need type conversion.

In [4]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Order ID                1010 non-null   int64  
 1   Customer ID             1010 non-null   object 
 2   Restaurant ID           1010 non-null   object 
 3   Order Date and Time     1010 non-null   object 
 4   Delivery Date and Time  1010 non-null   object 
 5   Order Value             969 non-null    float64
 6   Delivery Fee            960 non-null    float64
 7   Payment Method          960 non-null    object 
 8   Discounts and Offers    768 non-null    object 
 9   Commission Fee          970 non-null    float64
 10  Payment Processing Fee  959 non-null    float64
 11  Refunds/Chargebacks     970 non-null    float64
 12  Weather Condition       1010 non-null   object 
 13  Order Channel           1010 non-null   object 
dtypes: float64(5), int64(1), object(8)
memor

### 1.4 Summarise numeric and categorical columns

`describe()` gives statistical summaries for numeric fields. Using `include='object'` gives counts and most frequent values for text/category fields.

In [5]:
print("Numeric column summary")
display(raw_df.describe())

print("Categorical/text column summary")
display(raw_df.describe(include=["object"]))

Numeric column summary


,Order ID,Order Value,Delivery Fee,Commission Fee,Payment Processing Fee,Refunds/Chargebacks
count,1010.000000,969.000000,960.000000,970.000000,959.000000,970.000000
mean,499.221782,1055.901961,28.625000,127.042268,29.895725,27.886598
std,288.589479,529.534182,16.927281,43.013345,11.731583,49.383898
min,1.000000,104.000000,0.000000,50.000000,10.000000,0.000000
25%,248.250000,600.000000,20.000000,90.250000,20.000000,0.000000
50%,498.500000,1041.000000,30.000000,128.000000,30.000000,0.000000
75%,748.750000,1494.000000,40.000000,164.000000,40.000000,50.000000
max,1000.000000,1995.000000,50.000000,200.000000,50.000000,150.000000


Categorical/text column summary


,Customer ID,Restaurant ID,Order Date and Time,Delivery Date and Time,Payment Method,Discounts and Offers,Weather Condition,Order Channel
count,1010,1010,1010,1010,960,768,1010,1010
unique,947,621,1000,1000,3,4,4,3
top,C7949,R2317,2024-01-20 09:37:53,2024-01-20 11:05:53,Cash on Delivery,10%,Sunny,Mobile App
freq,3,6,2,2,341,221,405,693


### 1.5 Describe the purpose and expected type of each column

The descriptions below are based on the column names and values in this food-orders dataset. The **current type** is what Pandas detected when reading the CSV. The **expected type** describes how the field should be treated during analysis.

In [6]:
column_meanings = {
    "Order ID": ("Unique identifier assigned to an order", "Identifier"),
    "Customer ID": ("Identifier of the customer who placed the order", "Identifier"),
    "Restaurant ID": ("Identifier of the restaurant receiving the order", "Identifier"),
    "Order Date and Time": ("Date and time when the customer placed the order", "Datetime"),
    "Delivery Date and Time": ("Date and time when the order was delivered", "Datetime"),
    "Order Value": ("Monetary value of the food order", "Numeric"),
    "Delivery Fee": ("Fee charged for delivery", "Numeric"),
    "Payment Method": ("Method used by the customer to pay", "Category"),
    "Discounts and Offers": ("Discount or promotional offer recorded for the order", "Category"),
    "Commission Fee": ("Commission amount associated with the order", "Numeric"),
    "Payment Processing Fee": ("Fee for processing the customer's payment", "Numeric"),
    "Refunds/Chargebacks": ("Refund or chargeback amount recorded for the order", "Numeric"),
    "Weather Condition": ("Weather condition associated with the order", "Category"),
    "Order Channel": ("Platform or channel through which the order was placed", "Category")
}

description_rows = []
for column in raw_df.columns:
    meaning, expected_type = column_meanings.get(
        column,
        ("Meaning should be confirmed from dataset documentation", "Review required")
    )
    description_rows.append({
        "Column": column,
        "What it appears to contain": meaning,
        "Current Pandas Type": str(raw_df[column].dtype),
        "Expected Type": expected_type
    })

column_description = pd.DataFrame(description_rows)
display(column_description)

,Column,What it appears to contain,Current Pandas Type,Expected Type
0,Order ID,Unique identifier assigned to an order,int64,Identifier
1,Customer ID,Identifier of the customer who placed the order,object,Identifier
2,Restaurant ID,Identifier of the restaurant receiving the order,object,Identifier
3,Order Date and Time,Date and time when the customer placed the order,object,Datetime
4,Delivery Date and Time,Date and time when the order was delivered,object,Datetime
5,Order Value,Monetary value of the food order,float64,Numeric
6,Delivery Fee,Fee charged for delivery,float64,Numeric
7,Payment Method,Method used by the customer to pay,object,Category
8,Discounts and Offers,Discount or promotional offer recorded for the...,object,Category
9,Commission Fee,Commission amount associated with the order,float64,Numeric


### Task 1 — Observation

One row represents **one food order transaction** recorded by the food-delivery company. The order identifier helps distinguish orders, while customer and restaurant identifiers help analyse ordering behaviour and restaurant activity.

- **Order value:** `Order Value` can be used to study the value of customer orders.
- **Delivery performance:** `Order Date and Time` and `Delivery Date and Time` can be used to calculate delivery duration.
- **Payments:** `Payment Method`, `Payment Processing Fee` and `Refunds/Chargebacks` help understand payment activity.
- **Platform revenue:** `Commission Fee` and `Delivery Fee`, together with processing fees and refunds/chargebacks, are needed for the simplified platform net revenue measure in Task 5.
- **Business comparisons:** `Order Channel`, `Weather Condition` and `Discounts and Offers` can help compare order patterns across categories.

These are initial interpretations based on the available column names. Confirm field meanings with the business or dataset documentation where necessary.

# Task 2 — Perform a data-quality assessment

In this task, I check the original dataset for issues that may affect analysis. I do these checks on `raw_df` so that the original data remains unchanged. The results will guide the cleaning rules in Task 3.

### 2.1 Missing values and missing percentages

The count shows how many values are missing in each column. The percentage makes it easier to compare columns even when the dataset size changes.

In [7]:
missing_summary = pd.DataFrame({
    "Missing Values": raw_df.isna().sum(),
    "Missing Percentage": (raw_df.isna().mean() * 100).round(2)
})
display(missing_summary.sort_values("Missing Percentage", ascending=False))

,Missing Values,Missing Percentage
Discounts and Offers,242,23.96
Payment Processing Fee,51,5.05
Delivery Fee,50,4.95
Payment Method,50,4.95
Order Value,41,4.06
Commission Fee,40,3.96
Refunds/Chargebacks,40,3.96
Delivery Date and Time,0,0.00
Restaurant ID,0,0.00
Customer ID,0,0.00


### 2.2 Find fully duplicated rows

A fully duplicated row has the same value in every column as another row. I inspect these records before deciding whether to remove them.

In [8]:
exact_duplicate_count = raw_df.duplicated().sum()
print("Fully duplicated rows (additional copies):", exact_duplicate_count)
print("All rows involved in exact duplicates:")
display(raw_df[raw_df.duplicated(keep=False)].sort_values("Order ID"))

Fully duplicated rows (additional copies): 10
All rows involved in exact duplicates:


,Order ID,Customer ID,Restaurant ID,Order Date and Time,Delivery Date and Time,Order Value,Delivery Fee,Payment Method,Discounts and Offers,Commission Fee,Payment Processing Fee,Refunds/Chargebacks,Weather Condition,Order Channel
121,122,C7019,R2277,2024-02-05 05:55:15,2024-02-05 07:33:15,1438.0,0.0,Cash on Delivery,50 off Promo,51.0,NaN,0.0,Cloudy,Mobile App
1003,122,C7019,R2277,2024-02-05 05:55:15,2024-02-05 07:33:15,1438.0,0.0,Cash on Delivery,50 off Promo,51.0,NaN,0.0,Cloudy,Mobile App
132,133,C6915,R2074,2024-01-20 09:37:53,2024-01-20 11:05:53,1137.0,40.0,Credit Card,NaN,50.0,50.0,150.0,Cloudy,Mobile App
1009,133,C6915,R2074,2024-01-20 09:37:53,2024-01-20 11:05:53,1137.0,40.0,Credit Card,NaN,50.0,50.0,150.0,Cloudy,Mobile App
1004,156,C2853,R2286,2024-01-29 03:43:13,2024-01-29 04:32:13,1017.0,40.0,Cash on Delivery,5% on App,95.0,19.0,0.0,Rainy,Mobile App
155,156,C2853,R2286,2024-01-29 03:43:13,2024-01-29 04:32:13,1017.0,40.0,Cash on Delivery,5% on App,95.0,19.0,0.0,Rainy,Mobile App
1008,197,C2986,R2097,2024-01-16 13:11:39,2024-01-16 14:04:39,841.0,0.0,Cash on Delivery,15% New User,105.0,42.0,50.0,Cloudy,Mobile App
196,197,C2986,R2097,2024-01-16 13:11:39,2024-01-16 14:04:39,841.0,0.0,Cash on Delivery,15% New User,105.0,42.0,50.0,Cloudy,Mobile App
207,208,C5736,R2750,2024-01-18 09:08:01,2024-01-18 09:49:01,NaN,50.0,Digital Wallet,50 off Promo,184.0,41.0,0.0,Rainy,Mobile App
1005,208,C5736,R2750,2024-01-18 09:08:01,2024-01-18 09:49:01,NaN,50.0,Digital Wallet,50 off Promo,184.0,41.0,0.0,Rainy,Mobile App


### 2.3 Check repeated Order ID values

A repeated Order ID is not automatically proof of two different transactions. I compare the complete rows first. If repeated IDs are exact copies, they will be handled as exact duplicates. If the rows differ, they must be retained for investigation rather than deleted without evidence.

In [9]:
if "Order ID" in raw_df.columns:
    repeated_order_rows = raw_df[
        raw_df["Order ID"].duplicated(keep=False)
    ].sort_values("Order ID")

    print("Rows having an Order ID that appears more than once:",
          len(repeated_order_rows))
    print("Number of distinct repeated Order IDs:",
          repeated_order_rows["Order ID"].nunique())
    display(repeated_order_rows)
else:
    print("Order ID column was not found. Please verify the CSV column name.")

Rows having an Order ID that appears more than once: 20
Number of distinct repeated Order IDs: 10


,Order ID,Customer ID,Restaurant ID,Order Date and Time,Delivery Date and Time,Order Value,Delivery Fee,Payment Method,Discounts and Offers,Commission Fee,Payment Processing Fee,Refunds/Chargebacks,Weather Condition,Order Channel
121,122,C7019,R2277,2024-02-05 05:55:15,2024-02-05 07:33:15,1438.0,0.0,Cash on Delivery,50 off Promo,51.0,NaN,0.0,Cloudy,Mobile App
1003,122,C7019,R2277,2024-02-05 05:55:15,2024-02-05 07:33:15,1438.0,0.0,Cash on Delivery,50 off Promo,51.0,NaN,0.0,Cloudy,Mobile App
132,133,C6915,R2074,2024-01-20 09:37:53,2024-01-20 11:05:53,1137.0,40.0,Credit Card,NaN,50.0,50.0,150.0,Cloudy,Mobile App
1009,133,C6915,R2074,2024-01-20 09:37:53,2024-01-20 11:05:53,1137.0,40.0,Credit Card,NaN,50.0,50.0,150.0,Cloudy,Mobile App
1004,156,C2853,R2286,2024-01-29 03:43:13,2024-01-29 04:32:13,1017.0,40.0,Cash on Delivery,5% on App,95.0,19.0,0.0,Rainy,Mobile App
155,156,C2853,R2286,2024-01-29 03:43:13,2024-01-29 04:32:13,1017.0,40.0,Cash on Delivery,5% on App,95.0,19.0,0.0,Rainy,Mobile App
1008,197,C2986,R2097,2024-01-16 13:11:39,2024-01-16 14:04:39,841.0,0.0,Cash on Delivery,15% New User,105.0,42.0,50.0,Cloudy,Mobile App
196,197,C2986,R2097,2024-01-16 13:11:39,2024-01-16 14:04:39,841.0,0.0,Cash on Delivery,15% New User,105.0,42.0,50.0,Cloudy,Mobile App
207,208,C5736,R2750,2024-01-18 09:08:01,2024-01-18 09:49:01,NaN,50.0,Digital Wallet,50 off Promo,184.0,41.0,0.0,Rainy,Mobile App
1005,208,C5736,R2750,2024-01-18 09:08:01,2024-01-18 09:49:01,NaN,50.0,Digital Wallet,50 off Promo,184.0,41.0,0.0,Rainy,Mobile App


### 2.4 Review data types and identify columns that need conversion

For this dataset, the two date-and-time columns should be datetime values. Financial fields should be numeric. Identifier columns may be stored as text because they are labels, not quantities to calculate.

In [10]:
dtype_review = pd.DataFrame({
    "Column": raw_df.columns,
    "Current Data Type": [str(raw_df[col].dtype) for col in raw_df.columns]
})
display(dtype_review)

expected_timestamp_columns = [
    "Order Date and Time",
    "Delivery Date and Time"
]
expected_financial_columns = [
    "Order Value",
    "Delivery Fee",
    "Commission Fee",
    "Payment Processing Fee",
    "Refunds/Chargebacks"
]

print("Timestamp columns and their current types:")
for col in expected_timestamp_columns:
    if col in raw_df.columns:
        print(f"{col}: {raw_df[col].dtype}")

print("\nFinancial columns and their current types:")
for col in expected_financial_columns:
    if col in raw_df.columns:
        print(f"{col}: {raw_df[col].dtype}")

,Column,Current Data Type
0,Order ID,int64
1,Customer ID,object
2,Restaurant ID,object
3,Order Date and Time,object
4,Delivery Date and Time,object
5,Order Value,float64
6,Delivery Fee,float64
7,Payment Method,object
8,Discounts and Offers,object
9,Commission Fee,float64


Timestamp columns and their current types:
Order Date and Time: object
Delivery Date and Time: object

Financial columns and their current types:
Order Value: float64
Delivery Fee: float64
Commission Fee: float64
Payment Processing Fee: float64
Refunds/Chargebacks: float64


### 2.5 Check whether timestamps can be parsed

I use `pd.to_datetime(..., errors='coerce')` for assessment only. Valid timestamps are converted to datetime values. Any non-missing value that cannot be interpreted becomes `NaT` in this temporary result. The original columns in `raw_df` are not changed.

In [11]:
order_col = "Order Date and Time"
delivery_col = "Delivery Date and Time"

if order_col in raw_df.columns and delivery_col in raw_df.columns:
    order_time_check = pd.to_datetime(raw_df[order_col], errors="coerce")
    delivery_time_check = pd.to_datetime(raw_df[delivery_col], errors="coerce")

    timestamp_quality = pd.DataFrame({
        "Timestamp Column": [order_col, delivery_col],
        "Original Missing Values": [
            raw_df[order_col].isna().sum(),
            raw_df[delivery_col].isna().sum()
        ],
        "Non-missing Values That Failed Parsing": [
            (raw_df[order_col].notna() & order_time_check.isna()).sum(),
            (raw_df[delivery_col].notna() & delivery_time_check.isna()).sum()
        ],
        "Successfully Parsed Values": [
            order_time_check.notna().sum(),
            delivery_time_check.notna().sum()
        ]
    })
    display(timestamp_quality)
else:
    print("One or both timestamp columns were not found. Check the CSV column names.")

,Timestamp Column,Original Missing Values,Non-missing Values That Failed Parsing,Successfully Parsed Values
0,Order Date and Time,0,0,1010
1,Delivery Date and Time,0,0,1010


### 2.6 Find deliveries recorded before the order

A delivery timestamp earlier than its order timestamp is logically inconsistent. I list such records for investigation.

In [12]:
if order_col in raw_df.columns and delivery_col in raw_df.columns:
    invalid_sequence_mask = (
        order_time_check.notna()
        & delivery_time_check.notna()
        & (delivery_time_check < order_time_check)
    )
    invalid_delivery_sequence = raw_df.loc[invalid_sequence_mask].copy()

    print("Records where delivery occurs before the order:",
          len(invalid_delivery_sequence))
    display(invalid_delivery_sequence)

Records where delivery occurs before the order: 0


,Order ID,Customer ID,Restaurant ID,Order Date and Time,Delivery Date and Time,Order Value,Delivery Fee,Payment Method,Discounts and Offers,Commission Fee,Payment Processing Fee,Refunds/Chargebacks,Weather Condition,Order Channel


### 2.7 Check financial values and delivery duration

Negative financial values may be valid in some accounting systems (for example, adjustments), so I identify them rather than automatically deleting them. For delivery duration, I calculate a temporary difference and flag negative values and durations above 120 minutes for review. The 120-minute value is an investigation threshold for this exercise, not proof that every longer delivery is incorrect.

In [13]:
financial_check = []
for col in expected_financial_columns:
    if col in raw_df.columns:
        numeric_values = pd.to_numeric(raw_df[col], errors="coerce")
        financial_check.append({
            "Financial Column": col,
            "Negative Values": (numeric_values < 0).sum(),
            "Zero Values": (numeric_values == 0).sum(),
            "Missing Values": numeric_values.isna().sum()
        })

display(pd.DataFrame(financial_check))

if order_col in raw_df.columns and delivery_col in raw_df.columns:
    temporary_duration = (
        delivery_time_check - order_time_check
    ).dt.total_seconds() / 60

    print("Negative delivery durations:", (temporary_duration < 0).sum())
    print("Zero delivery durations:", (temporary_duration == 0).sum())
    print("Durations above 120 minutes:", (temporary_duration > 120).sum())
    print("Delivery duration summary (minutes):")
    display(temporary_duration.describe())

,Financial Column,Negative Values,Zero Values,Missing Values
0,Order Value,0,0,41
1,Delivery Fee,0,179,50
2,Commission Fee,0,0,40
3,Payment Processing Fee,0,0,51
4,Refunds/Chargebacks,0,697,40


Negative delivery durations: 0
Zero delivery durations: 0
Durations above 120 minutes: 0
Delivery duration summary (minutes):


count    1010.000000
mean       73.606931
std        25.970574
min        30.000000
25%        50.000000
50%        74.000000
75%        96.000000
max       119.000000
dtype: float64

### 2.8 Review categories and their frequency counts

Frequency counts show the values present in important categorical columns. `dropna=False` ensures missing categories are visible rather than hidden.

In [14]:
categorical_columns = [
    "Payment Method",
    "Discounts and Offers",
    "Weather Condition",
    "Order Channel"
]

for col in categorical_columns:
    print(f"\n{col}")
    if col in raw_df.columns:
        print("Distinct values:")
        display(pd.Series(raw_df[col].dropna().unique(), name=col))
        print("Frequency counts:")
        display(raw_df[col].value_counts(dropna=False).rename("Count").to_frame())
    else:
        print("Column not found. Check the column name.")


Payment Method
Distinct values:


0         Credit Card
1      Digital Wallet
2    Cash on Delivery
Name: Payment Method, dtype: object

Frequency counts:


,Count
Payment Method,
Cash on Delivery,341
Credit Card,323
Digital Wallet,296
NaN,50



Discounts and Offers
Distinct values:


0       5% on App
1             10%
2    15% New User
3    50 off Promo
Name: Discounts and Offers, dtype: object

... (6 outputs left out)


Order Channel
Distinct values:


0     Mobile App
1    Partner App
2        Website
Name: Order Channel, dtype: object

Frequency counts:


,Count
Order Channel,
Mobile App,693
Partner App,165
Website,152


### Task 2 — Data-quality observations

After running the checks, review the output and write down at least three issues that are actually present in this dataset. The examples below explain what to look for; keep only statements supported by your results.

1. **Missing values in financial fields:** missing order values or fees reduce the number of records available for financial calculations. Treating missing amounts as zero could distort business measures.
2. **Exact duplicate rows or repeated Order IDs:** duplicate transactions can inflate order counts and financial totals. Repeated IDs that are not exact copies need investigation.
3. **Timestamp quality:** invalid timestamps or delivery times earlier than order times make delivery-duration calculations unreliable.
4. **Missing payment or offer categories:** missing category details make it harder to compare payment methods and promotional activity.
5. **Unusual financial or delivery values:** negative amounts or unusually long durations should be reviewed before they are used in reports.


# Task 3 — Design cleaning rules before changing data

Before modifying the working data, I decide how each issue should be handled. These rules prevent assumptions from being made during cleaning and help keep the results consistent.

| Data-quality issue | Chosen handling rule | Reason / business assumption |
|---|---|---|
| Exact duplicate rows | Inspect them, then remove fully identical extra copies in `work_df`. | Exact copies can inflate order counts and financial totals. |
| Repeated Order ID values | After removing exact duplicates, check repeated IDs again. Keep non-identical records and flag them for investigation. | A repeated identifier may indicate a data issue, but it is not enough evidence to delete a transaction. |
| Missing Order Value | Retain as missing. Exclude these records from calculations that require Order Value. | A missing order value is unknown, not zero. |
| Missing Delivery Fee | Retain as missing. | Missing does not prove that delivery was free. |
| Missing Commission Fee | Retain as missing. | Replacing it with zero could understate commission revenue. |
| Missing Payment Processing Fee | Retain as missing. | Missing does not mean that no processing fee was charged. |
| Missing Refunds/Chargebacks | Retain as missing. | Missing refund information does not prove that no refund or chargeback occurred. |
| Missing Payment Method | Replace with `Unknown`. | This makes unavailable payment information visible without guessing the method. |
| Missing Discounts and Offers | Replace with `Not Specified`. | This does not incorrectly claim that no discount was applied. |
| Invalid timestamps | Convert with `errors='coerce'`; invalid values become `NaT` and are reported. | Invalid date strings cannot be used reliably in time calculations. |
| Delivery before order | Retain the order and flag its negative duration for investigation. | The record may need correction at source; deleting it would lose transaction information. |
| Delivery duration above 120 minutes | Retain and flag for investigation; exclude from valid-duration summaries when appropriate. | It may be a genuine delay or a data issue. The threshold is an exercise rule, not an automatic deletion rule. |
| Negative financial values | Flag for review rather than automatically changing or deleting them. | Some financial systems may use negative values for adjustments or reversals. |

### Cleaning assumptions

- I will make all changes to `work_df`, not `raw_df`.
- I will remove only exact duplicate rows.
- I will not remove non-identical rows just because their Order ID is repeated.
- Missing financial values will remain missing.
- `Unknown` and `Not Specified` are labels for unavailable category information; they are not actual payment or offer values.
- Suspicious durations and financial values will be flagged for review.

# Task 4 — Clean the dataset

I now apply the rules documented in Task 3. I record the row counts before and after duplicate removal so that the effect of cleaning is clear.

### 4.1 Create a separate working copy

In [15]:
work_df = raw_df.copy()

original_row_count = len(raw_df)
print("Original dataset shape:", raw_df.shape)
print("Working dataset shape before cleaning:", work_df.shape)

Original dataset shape: (1010, 14)
Working dataset shape before cleaning: (1010, 14)


### 4.2 Inspect and remove fully duplicated rows

I first count and display exact duplicates. Then I remove only the extra identical copies. `keep='first'` keeps the first occurrence.

In [16]:
duplicate_rows_before = work_df.duplicated().sum()
print("Exact duplicate rows before removal:", duplicate_rows_before)

display(work_df[work_df.duplicated(keep=False)].sort_values("Order ID"))

work_df = work_df.drop_duplicates(keep="first").copy()

duplicate_rows_after = work_df.duplicated().sum()
rows_removed_as_duplicates = original_row_count - len(work_df)

print("Rows after removing exact duplicates:", len(work_df))
print("Rows removed:", rows_removed_as_duplicates)
print("Exact duplicate rows remaining:", duplicate_rows_after)

Exact duplicate rows before removal: 10


,Order ID,Customer ID,Restaurant ID,Order Date and Time,Delivery Date and Time,Order Value,Delivery Fee,Payment Method,Discounts and Offers,Commission Fee,Payment Processing Fee,Refunds/Chargebacks,Weather Condition,Order Channel
121,122,C7019,R2277,2024-02-05 05:55:15,2024-02-05 07:33:15,1438.0,0.0,Cash on Delivery,50 off Promo,51.0,NaN,0.0,Cloudy,Mobile App
1003,122,C7019,R2277,2024-02-05 05:55:15,2024-02-05 07:33:15,1438.0,0.0,Cash on Delivery,50 off Promo,51.0,NaN,0.0,Cloudy,Mobile App
132,133,C6915,R2074,2024-01-20 09:37:53,2024-01-20 11:05:53,1137.0,40.0,Credit Card,NaN,50.0,50.0,150.0,Cloudy,Mobile App
1009,133,C6915,R2074,2024-01-20 09:37:53,2024-01-20 11:05:53,1137.0,40.0,Credit Card,NaN,50.0,50.0,150.0,Cloudy,Mobile App
1004,156,C2853,R2286,2024-01-29 03:43:13,2024-01-29 04:32:13,1017.0,40.0,Cash on Delivery,5% on App,95.0,19.0,0.0,Rainy,Mobile App
155,156,C2853,R2286,2024-01-29 03:43:13,2024-01-29 04:32:13,1017.0,40.0,Cash on Delivery,5% on App,95.0,19.0,0.0,Rainy,Mobile App
1008,197,C2986,R2097,2024-01-16 13:11:39,2024-01-16 14:04:39,841.0,0.0,Cash on Delivery,15% New User,105.0,42.0,50.0,Cloudy,Mobile App
196,197,C2986,R2097,2024-01-16 13:11:39,2024-01-16 14:04:39,841.0,0.0,Cash on Delivery,15% New User,105.0,42.0,50.0,Cloudy,Mobile App
207,208,C5736,R2750,2024-01-18 09:08:01,2024-01-18 09:49:01,NaN,50.0,Digital Wallet,50 off Promo,184.0,41.0,0.0,Rainy,Mobile App
1005,208,C5736,R2750,2024-01-18 09:08:01,2024-01-18 09:49:01,NaN,50.0,Digital Wallet,50 off Promo,184.0,41.0,0.0,Rainy,Mobile App


Rows after removing exact duplicates: 1000
Rows removed: 10
Exact duplicate rows remaining: 0


### 4.3 Check repeated Order IDs after exact duplicate removal

At this stage, any repeated Order IDs are not fully identical rows. I keep these records and display them for investigation.

In [17]:
if "Order ID" in work_df.columns:
    repeated_order_mask = work_df["Order ID"].duplicated(keep=False)
    repeated_order_ids = work_df.loc[repeated_order_mask].sort_values("Order ID")

    print("Rows with non-unique Order IDs after exact duplicate removal:",
          len(repeated_order_ids))
    print("Distinct Order IDs involved:",
          repeated_order_ids["Order ID"].nunique())
    display(repeated_order_ids)
else:
    print("Order ID column not found.")

Rows with non-unique Order IDs after exact duplicate removal: 0
Distinct Order IDs involved: 0


,Order ID,Customer ID,Restaurant ID,Order Date and Time,Delivery Date and Time,Order Value,Delivery Fee,Payment Method,Discounts and Offers,Commission Fee,Payment Processing Fee,Refunds/Chargebacks,Weather Condition,Order Channel


### 4.4 Convert timestamp columns

I convert both timestamp fields to Pandas datetime format. `errors='coerce'` converts invalid non-missing date strings to `NaT`. I count parsing failures before and after conversion so these records can be identified.

In [18]:
timestamp_columns = [
    "Order Date and Time",
    "Delivery Date and Time"
]

timestamp_parse_issues = {}

for col in timestamp_columns:
    if col in work_df.columns:
        parsed_values = pd.to_datetime(work_df[col], errors="coerce")
        failed_count = (work_df[col].notna() & parsed_values.isna()).sum()

        work_df[col] = parsed_values
        timestamp_parse_issues[col] = failed_count

        print(f"{col}:")
        print("  Parsing failures:", failed_count)
        print("  Missing after conversion:", work_df[col].isna().sum())
        print("  New data type:", work_df[col].dtype)

Order Date and Time:
  Parsing failures: 0
  Missing after conversion: 0
  New data type: datetime64[ns]
Delivery Date and Time:
  Parsing failures: 0
  Missing after conversion: 0
  New data type: datetime64[ns]


### 4.5 Convert financial columns to numeric

Financial fields must be numeric for totals and other calculations. I use `pd.to_numeric(errors='coerce')`; values that cannot be converted become missing instead of being treated as zero. I retain existing missing values.

In [19]:
financial_columns = [
    "Order Value",
    "Delivery Fee",
    "Commission Fee",
    "Payment Processing Fee",
    "Refunds/Chargebacks"
]

numeric_conversion_issues = {}

for col in financial_columns:
    if col in work_df.columns:
        converted_values = pd.to_numeric(work_df[col], errors="coerce")
        failed_count = (work_df[col].notna() & converted_values.isna()).sum()

        work_df[col] = converted_values
        numeric_conversion_issues[col] = failed_count

        print(f"{col}:")
        print("  Values that failed numeric conversion:", failed_count)
        print("  Data type after conversion:", work_df[col].dtype)

Order Value:
  Values that failed numeric conversion: 0
  Data type after conversion: float64
Delivery Fee:
  Values that failed numeric conversion: 0
  Data type after conversion: float64
Commission Fee:
  Values that failed numeric conversion: 0
  Data type after conversion: float64
Payment Processing Fee:
  Values that failed numeric conversion: 0
  Data type after conversion: float64
Refunds/Chargebacks:
  Values that failed numeric conversion: 0
  Data type after conversion: float64


### 4.6 Handle missing categorical values

I fill missing payment methods with `Unknown` and missing offer details with `Not Specified`. These labels make the missing information explicit. I do not use `No Discount`, because a blank offer field does not confirm that no offer was used.

In [20]:
if "Payment Method" in work_df.columns:
    work_df["Payment Method"] = work_df["Payment Method"].fillna("Unknown")

if "Discounts and Offers" in work_df.columns:
    work_df["Discounts and Offers"] = (
        work_df["Discounts and Offers"].fillna("Not Specified")
    )

print("Missing Payment Method values:",
      work_df["Payment Method"].isna().sum()
      if "Payment Method" in work_df.columns else "Column not found")

print("Missing Discounts and Offers values:",
      work_df["Discounts and Offers"].isna().sum()
      if "Discounts and Offers" in work_df.columns else "Column not found")

Missing Payment Method values: 0
Missing Discounts and Offers values: 0


### 4.7 Retain missing financial values and review suspicious values

I do not fill missing financial values. A missing fee or refund is not necessarily zero. I also check for negative financial values and keep them for investigation rather than changing them without business confirmation.

In [21]:
available_financial_columns = [
    col for col in financial_columns if col in work_df.columns
]

print("Missing financial values after cleaning:")
display(work_df[available_financial_columns].isna().sum().to_frame("Missing Count"))

negative_financial_summary = []
for col in available_financial_columns:
    negative_financial_summary.append({
        "Column": col,
        "Negative Values": (work_df[col] < 0).sum()
    })

display(pd.DataFrame(negative_financial_summary))

Missing financial values after cleaning:


,Missing Count
Order Value,40
Delivery Fee,50
Commission Fee,40
Payment Processing Fee,50
Refunds/Chargebacks,40


,Column,Negative Values
0,Order Value,0
1,Delivery Fee,0
2,Commission Fee,0
3,Payment Processing Fee,0
4,Refunds/Chargebacks,0


### 4.8 Record row counts and inspect the cleaned dataset

Only exact duplicate rows have been removed in this task. Other records are retained, including records with missing financial values or suspicious timestamps, so they can be investigated and handled transparently.

In [22]:
cleaned_row_count = len(work_df)

print("Original rows:", original_row_count)
print("Rows after cleaning:", cleaned_row_count)
print("Rows removed:", original_row_count - cleaned_row_count)
print("Rows retained:", cleaned_row_count)
print("Exact duplicates remaining:", work_df.duplicated().sum())

print("\nFirst five rows of cleaned data:")
display(work_df.head())

print("\nData types after cleaning:")
display(work_df.dtypes.to_frame("Data Type"))

Original rows: 1010
Rows after cleaning: 1000
Rows removed: 10
Rows retained: 1000
Exact duplicates remaining: 0

First five rows of cleaned data:


,Order ID,Customer ID,Restaurant ID,Order Date and Time,Delivery Date and Time,Order Value,Delivery Fee,Payment Method,Discounts and Offers,Commission Fee,Payment Processing Fee,Refunds/Chargebacks,Weather Condition,Order Channel
0,1,C8270,R2924,2024-02-01 01:11:52,2024-02-01 02:39:52,1914.0,0.0,Credit Card,5% on App,150.0,47.0,0.0,Rainy,Mobile App
1,2,C1860,R2054,2024-02-02 22:11:04,2024-02-02 22:46:04,986.0,40.0,Digital Wallet,10%,198.0,23.0,0.0,Cloudy,Mobile App
2,3,C6390,R2870,2024-01-31 05:54:35,2024-01-31 06:52:35,937.0,30.0,Cash on Delivery,15% New User,195.0,45.0,0.0,Humid,Mobile App
3,4,C6191,R2642,2024-01-16 22:52:49,2024-01-16 23:38:49,1463.0,NaN,Cash on Delivery,Not Specified,146.0,27.0,0.0,Rainy,Mobile App
4,5,C6734,R2799,2024-01-29 01:19:30,2024-01-29 02:48:30,1992.0,30.0,Cash on Delivery,50 off Promo,130.0,50.0,NaN,Sunny,Mobile App



Data types after cleaning:


,Data Type
Order ID,int64
Customer ID,object
Restaurant ID,object
Order Date and Time,datetime64[ns]
Delivery Date and Time,datetime64[ns]
Order Value,float64
Delivery Fee,float64
Payment Method,object
Discounts and Offers,object
Commission Fee,float64


### 4.9 Review missing values that remain

Some missing values are intentionally retained, especially financial fields and invalid timestamps. This summary helps explain what is still unavailable after cleaning.

In [23]:
remaining_missing_summary = pd.DataFrame({
    "Missing Values": work_df.isna().sum(),
    "Missing Percentage": (work_df.isna().mean() * 100).round(2)
})

display(
    remaining_missing_summary[
        remaining_missing_summary["Missing Values"] > 0
    ].sort_values("Missing Values", ascending=False)
)

,Missing Values,Missing Percentage
Delivery Fee,50,5.0
Payment Processing Fee,50,5.0
Order Value,40,4.0
Commission Fee,40,4.0
Refunds/Chargebacks,40,4.0


### Task 4 — Cleaning observations

After running the cells, record the actual results:

- **Rows remaining:** use the `Rows after cleaning` value above.
- **Rows removed:** these should be exact duplicate copies only.
- **Missing values retained:** financial values remain missing because their actual amounts are unknown. Invalid timestamps remain `NaT` because they cannot be reliably parsed.
- **Repeated Order IDs:** any non-identical records with a repeated Order ID are retained for investigation rather than deleted automatically.


# Task 5 — Transform the data and create derived columns

In this task, I create four new columns from the cleaned data. These columns help analyse delivery performance, order timing and the platform's simplified net revenue.

### 5.1 Create `Delivery_Time_Minutes`

Delivery time is the difference between delivery timestamp and order timestamp. The result is expressed in minutes. If either timestamp is missing or invalid, the result remains missing.

In [24]:
work_df["Delivery_Time_Minutes"] = (
    work_df["Delivery Date and Time"] -
    work_df["Order Date and Time"]
).dt.total_seconds() / 60

display(work_df[
    ["Order ID", "Order Date and Time",
     "Delivery Date and Time", "Delivery_Time_Minutes"]
].head())

,Order ID,Order Date and Time,Delivery Date and Time,Delivery_Time_Minutes
0,1,2024-02-01 01:11:52,2024-02-01 02:39:52,88.0
1,2,2024-02-02 22:11:04,2024-02-02 22:46:04,35.0
2,3,2024-01-31 05:54:35,2024-01-31 06:52:35,58.0
3,4,2024-01-16 22:52:49,2024-01-16 23:38:49,46.0
4,5,2024-01-29 01:19:30,2024-01-29 02:48:30,89.0


### 5.2 Identify records without a delivery duration

A delivery duration cannot be calculated if the order timestamp or delivery timestamp is missing or invalid. Negative durations are calculated differences, but they are logically suspicious and must be flagged.

In [25]:
missing_duration_mask = work_df["Delivery_Time_Minutes"].isna()
missing_duration_records = work_df.loc[missing_duration_mask]

print("Records without a calculable delivery duration:",
      len(missing_duration_records))

display(missing_duration_records[
    ["Order ID", "Order Date and Time",
     "Delivery Date and Time", "Delivery_Time_Minutes"]
])

Records without a calculable delivery duration: 0


,Order ID,Order Date and Time,Delivery Date and Time,Delivery_Time_Minutes


### 5.3 Flag negative and unusually long delivery durations

A negative duration means the delivery timestamp is earlier than the order timestamp. For this exercise, I also flag durations above 120 minutes for investigation. I keep the calculated value and the order record so the issue is visible; I do not silently delete or overwrite it.

In [26]:
work_df["Delivery_Duration_Flag"] = np.select(
    [
        work_df["Delivery_Time_Minutes"].isna(),
        work_df["Delivery_Time_Minutes"] < 0,
        work_df["Delivery_Time_Minutes"] > 120
    ],
    [
        "Cannot Calculate",
        "Negative - Investigate",
        "Above 120 Minutes - Review"
    ],
    default="Within Exercise Range"
)

flagged_delivery_records = work_df[
    work_df["Delivery_Duration_Flag"].isin([
        "Negative - Investigate",
        "Above 120 Minutes - Review"
    ])
]

print("Negative durations:",
      (work_df["Delivery_Time_Minutes"] < 0).sum())
print("Durations above 120 minutes:",
      (work_df["Delivery_Time_Minutes"] > 120).sum())

display(flagged_delivery_records[
    ["Order ID", "Order Date and Time",
     "Delivery Date and Time", "Delivery_Time_Minutes",
     "Delivery_Duration_Flag"]
])

Negative durations: 0
Durations above 120 minutes: 0


,Order ID,Order Date and Time,Delivery Date and Time,Delivery_Time_Minutes,Delivery_Duration_Flag


### 5.4 Create `Order_Date`

I extract the date portion from the order timestamp. This is useful for grouping orders by day.

In [27]:
work_df["Order_Date"] = work_df["Order Date and Time"].dt.date

display(work_df[
    ["Order Date and Time", "Order_Date"]
].head())

,Order Date and Time,Order_Date
0,2024-02-01 01:11:52,2024-02-01
1,2024-02-02 22:11:04,2024-02-02
2,2024-01-31 05:54:35,2024-01-31
3,2024-01-16 22:52:49,2024-01-16
4,2024-01-29 01:19:30,2024-01-29


### 5.5 Create `Order_Hour`

I extract the hour from the order timestamp using the 24-hour clock (0 to 23). This can help identify busy ordering periods.

In [28]:
work_df["Order_Hour"] = work_df["Order Date and Time"].dt.hour

display(work_df[
    ["Order Date and Time", "Order_Hour"]
].head())

,Order Date and Time,Order_Hour
0,2024-02-01 01:11:52,1
1,2024-02-02 22:11:04,22
2,2024-01-31 05:54:35,5
3,2024-01-16 22:52:49,22
4,2024-01-29 01:19:30,1


### 5.6 Create `Platform_Net_Revenue`

For this assignment, I use the simplified formula:

**Platform Net Revenue = Commission Fee + Delivery Fee − Payment Processing Fee − Refunds/Chargebacks**

I calculate this value only when all four components are available. If one or more components are missing, the result remains missing. Filling a missing component with zero would assume that no fee or refund occurred, which may not be true.

In [29]:
revenue_components = [
    "Commission Fee",
    "Delivery Fee",
    "Payment Processing Fee",
    "Refunds/Chargebacks"
]

missing_revenue_columns = [
    col for col in revenue_components if col not in work_df.columns
]

if missing_revenue_columns:
    raise KeyError(f"Required revenue columns not found: {missing_revenue_columns}")

complete_revenue_mask = work_df[revenue_components].notna().all(axis=1)

work_df["Platform_Net_Revenue"] = np.nan

work_df.loc[complete_revenue_mask, "Platform_Net_Revenue"] = (
    work_df.loc[complete_revenue_mask, "Commission Fee"]
    + work_df.loc[complete_revenue_mask, "Delivery Fee"]
    - work_df.loc[complete_revenue_mask, "Payment Processing Fee"]
    - work_df.loc[complete_revenue_mask, "Refunds/Chargebacks"]
)

print("Records with all four revenue components:",
      complete_revenue_mask.sum())
print("Records missing one or more revenue components:",
      (~complete_revenue_mask).sum())
print("Platform net revenue values calculated:",
      work_df["Platform_Net_Revenue"].notna().sum())
print("Platform net revenue values left missing:",
      work_df["Platform_Net_Revenue"].isna().sum())

display(work_df[
    revenue_components + ["Platform_Net_Revenue"]
].head(10))

Records with all four revenue components: 828
Records missing one or more revenue components: 172
Platform net revenue values calculated: 828
Platform net revenue values left missing: 172


,Commission Fee,Delivery Fee,Payment Processing Fee,Refunds/Chargebacks,Platform_Net_Revenue
0,150.0,0.0,47.0,0.0,103.0
1,198.0,40.0,23.0,0.0,215.0
2,195.0,30.0,45.0,0.0,180.0
3,146.0,NaN,27.0,0.0,NaN
4,130.0,30.0,50.0,NaN,NaN
5,92.0,20.0,27.0,150.0,-65.0
6,144.0,30.0,12.0,50.0,112.0
7,55.0,0.0,19.0,0.0,36.0
8,116.0,40.0,48.0,0.0,108.0
9,189.0,40.0,10.0,0.0,219.0


### 5.7 Review the new columns and their data types

This final check confirms that all four requested derived columns have been created and shows how many values are missing in each.

In [30]:
new_columns = [
    "Delivery_Time_Minutes",
    "Order_Date",
    "Order_Hour",
    "Platform_Net_Revenue"
]

print("New columns:")
display(work_df[new_columns].head(10))

print("Data types of the new columns:")
display(work_df[new_columns].dtypes.to_frame("Data Type"))

print("Missing values in the new columns:")
display(work_df[new_columns].isna().sum().to_frame("Missing Count"))

New columns:


,Delivery_Time_Minutes,Order_Date,Order_Hour,Platform_Net_Revenue
0,88.0,2024-02-01,1,103.0
1,35.0,2024-02-02,22,215.0
2,58.0,2024-01-31,5,180.0
3,46.0,2024-01-16,22,NaN
4,89.0,2024-01-29,1,NaN
5,51.0,2024-01-25,4,-65.0
6,53.0,2024-01-12,23,112.0
7,110.0,2024-01-26,22,36.0
8,114.0,2024-01-02,18,108.0
9,80.0,2024-01-05,0,219.0


Data types of the new columns:


,Data Type
Delivery_Time_Minutes,float64
Order_Date,object
Order_Hour,int32
Platform_Net_Revenue,float64


Missing values in the new columns:


,Missing Count
Delivery_Time_Minutes,0
Order_Date,0
Order_Hour,0
Platform_Net_Revenue,172


### 5.8 Validate delivery duration and revenue calculations

For a basic delivery-time summary, I use only durations from 0 to 120 minutes. This is an exercise-specific analysis range; flagged records remain in the dataset for review. For revenue, only rows with all four components are included.

In [31]:
valid_delivery_times = work_df.loc[
    work_df["Delivery_Time_Minutes"].between(0, 120, inclusive="both"),
    "Delivery_Time_Minutes"
]

print("Valid delivery durations used in the summary:", len(valid_delivery_times))
if not valid_delivery_times.empty:
    print("Mean delivery time:", round(valid_delivery_times.mean(), 2), "minutes")
    print("Median delivery time:", round(valid_delivery_times.median(), 2), "minutes")
else:
    print("No delivery durations fall within the exercise range.")

valid_platform_revenue = work_df["Platform_Net_Revenue"].dropna()

print("\nRecords used for platform net revenue:", len(valid_platform_revenue))
print("Total simplified platform net revenue:",
      round(valid_platform_revenue.sum(), 2))

Valid delivery durations used in the summary: 1000
Mean delivery time: 73.58 minutes
Median delivery time: 74.0 minutes

Records used for platform net revenue: 828
Total simplified platform net revenue: 80877.0


### Task 5 — Questions and observations

**1. Which records cannot have a delivery duration calculated?**  
Records with a missing or invalid order timestamp, or a missing or invalid delivery timestamp, cannot have a duration calculated. Their `Delivery_Time_Minutes` value remains missing.

**2. Are any calculated delivery durations negative or implausible?**  
Check `Delivery_Duration_Flag` and the flagged-record table. Negative durations indicate an incorrect time sequence. Durations above 120 minutes are flagged for review under the exercise threshold. They are retained rather than silently accepted or deleted.

**3. Why should missing values in the revenue formula not automatically be replaced with zero?**  
A missing fee or refund does not prove that the actual amount was zero. Replacing it with zero could produce an incorrect platform revenue figure. Therefore, `Platform_Net_Revenue` is calculated only when all four required components are available.

### Task 5 — Transformation summary

The cleaned dataset now includes these derived columns:

- `Delivery_Time_Minutes` — time between order placement and delivery, in minutes.
- `Order_Date` — date on which the order was placed.
- `Order_Hour` — hour at which the order was placed.
- `Platform_Net_Revenue` — simplified platform net revenue, calculated only when all four financial components are present.
- `Delivery_Duration_Flag` — an additional review field to identify missing, negative or unusually long delivery durations.

The original `raw_df` remains unchanged. The cleaned and transformed data is stored in `work_df`.

# Task 6 — Answer the business questions 

### 6.1 Compare Original and Cleaned Dataset Records

In this step, I compare the number of records in the original dataset with the number of records remaining after cleaning.

This helps me understand how many records were removed during the cleaning process. In Task 4, I removed only fully duplicated rows and retained other records for analysis or investigation.

In [32]:
original_records = len(raw_df)
cleaned_records = len(work_df)

removed_records = original_records - cleaned_records

print("Original dataset records:", original_records)
print("Cleaned dataset records:", cleaned_records)
print("Records removed during cleaning:", removed_records)

Original dataset records: 1010
Cleaned dataset records: 1000
Records removed during cleaning: 10


### 6.2 Count Records with Valid Order Value

In this step, I identify the records that have a valid Order Value.

I exclude missing and negative Order Values because they cannot be treated as normal order amounts for this business calculation. Zero and positive values are retained.

In [33]:
valid_order_mask = (
    work_df["Order Value"].notna()
    & (work_df["Order Value"] >= 0)
)

valid_order_records = work_df.loc[valid_order_mask]

print("Total cleaned records:", len(work_df))
print("Records with valid Order Value:", len(valid_order_records))
print("Records excluded:", len(work_df) - len(valid_order_records))

Total cleaned records: 1000
Records with valid Order Value: 960
Records excluded: 40


### 6.3 Calculate Total, Mean and Median Order Value

In this step, I calculate the total, average and median Order Value using only valid order records.

The total shows the combined value of the included orders. The mean shows the average order value, while the median gives the middle value and helps understand a typical order without being affected as much by unusually high order values.

Records with missing or negative Order Values are excluded from these calculations.

In [34]:
total_order_value = valid_order_records["Order Value"].sum()

mean_order_value = valid_order_records["Order Value"].mean()

median_order_value = valid_order_records["Order Value"].median()

print("Number of valid orders:", len(valid_order_records))
print("Total Order Value:", round(total_order_value, 2))
print("Mean Order Value:", round(mean_order_value, 2))
print("Median Order Value:", round(median_order_value, 2))

Number of valid orders: 960
Total Order Value: 1013415.0
Mean Order Value: 1055.64
Median Order Value: 1040.0


### 6.4 Find the Most Frequently Used Payment Method

In this step, I check how many orders were placed using each payment method.

I exclude the Unknown category because it represents missing payment information. This helps me identify the payment method that customers actually used most frequently.

In [35]:
known_payment_methods = work_df[
    (work_df["Payment Method"] != "Unknown")
    & (work_df["Payment Method"].notna())
]

payment_counts = known_payment_methods["Payment Method"].value_counts()

print("Number of orders by payment method:")
display(payment_counts.to_frame("Order Count"))

print("Most frequently used payment method:")
print(payment_counts.idxmax())

print("Number of orders:")
print(payment_counts.max())

Number of orders by payment method:


,Order Count
Payment Method,
Cash on Delivery,336
Credit Card,320
Digital Wallet,294


Most frequently used payment method:
Cash on Delivery
Number of orders:
336


### 6.5 Find the Order Channel with the Highest Number of Orders

In this step, I count the orders received through each order channel.

I use value_counts() to compare the number of orders across different channels. Missing channel information is excluded because it cannot be assigned to a known channel.

In [36]:
known_channels = work_df[
    work_df["Order Channel"].notna()
    & (work_df["Order Channel"].astype(str).str.strip() != "")
]

channel_counts = known_channels["Order Channel"].value_counts()

print("Orders by channel:")
display(channel_counts.to_frame("Order Count"))

print("Order channel with the highest orders:")
print(channel_counts.idxmax())

print("Number of orders:")
print(channel_counts.max())

Orders by channel:


,Order Count
Order Channel,
Mobile App,686
Partner App,162
Website,152


Order channel with the highest orders:
Mobile App
Number of orders:
686


### 6.6 Find the Most Frequently Recorded Weather Condition

In this step, I analyse the Weather Condition column to understand which weather condition is recorded most frequently across the orders.

I exclude missing weather information and use value_counts() to find the most common condition.

In [37]:
known_weather = work_df[
    work_df["Weather Condition"].notna()
    & (work_df["Weather Condition"].astype(str).str.strip() != "")
]

weather_counts = known_weather["Weather Condition"].value_counts()

print("Orders by weather condition:")
display(weather_counts.to_frame("Order Count"))

print("Most frequent weather condition:")
print(weather_counts.idxmax())

print("Number of orders:")
print(weather_counts.max())

Orders by weather condition:


,Order Count
Weather Condition,
Sunny,402
Cloudy,247
Rainy,200
Humid,151


Most frequent weather condition:
Sunny
Number of orders:
402


### 6.7 Calculate Mean and Median Delivery Time

In this step, I calculate the average and median delivery time using the Delivery_Time_Minutes column created in Task 5.

I include only delivery durations between 0 and 120 minutes. Missing, negative and unusually long durations are excluded from this summary because they need further checking.

This helps the company understand the normal delivery time for the orders included in the calculation.

In [38]:
valid_delivery_times = work_df.loc[
    work_df["Delivery_Time_Minutes"].between(0, 120, inclusive="both")
]

mean_delivery_time = valid_delivery_times["Delivery_Time_Minutes"].mean()

median_delivery_time = valid_delivery_times["Delivery_Time_Minutes"].median()

print("Records with valid delivery duration:", len(valid_delivery_times))

print("Mean Delivery Time:",
      round(mean_delivery_time, 2), "minutes")

print("Median Delivery Time:",
      round(median_delivery_time, 2), "minutes")

print("Records excluded from delivery-time calculation:",
      len(work_df) - len(valid_delivery_times))

Records with valid delivery duration: 1000
Mean Delivery Time: 73.58 minutes
Median Delivery Time: 74.0 minutes
Records excluded from delivery-time calculation: 0


### 6.8 Find the Top Five Restaurants by Order Count

In this step, I identify the five restaurants that received the highest number of orders.

I group the records using Restaurant ID and count the number of orders for each restaurant. Then I sort the results in descending order and display the top five.

This helps the company understand which restaurants receive more orders through its platform.

In [39]:
restaurant_order_counts = (
    work_df
    .dropna(subset=["Restaurant ID"])
    .groupby("Restaurant ID")
    .size()
    .reset_index(name="Order Count")
)

top_five_restaurants = (
    restaurant_order_counts
    .sort_values("Order Count", ascending=False)
    .head(5)
    .reset_index(drop=True)
)

print("Top five restaurants by order count:")
display(top_five_restaurants)

Top five restaurants by order count:


,Restaurant ID,Order Count
0,R2317,6
1,R2726,5
2,R2523,5
3,R2083,5
4,R2016,5


### 6.9 Calculate Total Platform Net Revenue

In this step, I calculate the total Platform_Net_Revenue using the column created in Task 5.

I include only records where Commission Fee, Delivery Fee, Payment Processing Fee and Refunds/Chargebacks are all available.

I also count how many records are included in this calculation. Records with missing financial components are excluded because their actual revenue cannot be calculated accurately.

I do not replace missing financial values with zero because missing information does not mean that the actual amount was zero.

In [40]:
valid_platform_revenue = work_df[
    work_df["Platform_Net_Revenue"].notna()
]

total_platform_revenue = valid_platform_revenue[
    "Platform_Net_Revenue"
].sum()

included_revenue_records = len(valid_platform_revenue)

excluded_revenue_records = (
    len(work_df) - included_revenue_records
)

print("Records included in revenue calculation:",
      included_revenue_records)

print("Records excluded from revenue calculation:",
      excluded_revenue_records)

print("Total Platform Net Revenue:",
      round(total_platform_revenue, 2))

Records included in revenue calculation: 828
Records excluded from revenue calculation: 172
Total Platform Net Revenue: 80877.0


### 6.10 Final Business Analysis Summary

In this section, I bring together the important results from all the previous calculations.

This summary gives a clear view of the dataset, order values, customer payment preferences, order channels, weather conditions, delivery performance, restaurant orders and platform revenue.

All values are calculated from the cleaned and transformed dataset using Pandas.

In [41]:
business_summary = pd.DataFrame({
    "Business Metric": [
        "Original Dataset Records",
        "Cleaned Dataset Records",
        "Records with Valid Order Value",
        "Total Order Value",
        "Mean Order Value",
        "Median Order Value",
        "Most Used Payment Method",
        "Most Used Order Channel",
        "Most Frequent Weather Condition",
        "Mean Delivery Time (Minutes)",
        "Median Delivery Time (Minutes)",
        "Records Used for Delivery Analysis",
        "Total Platform Net Revenue",
        "Records Used for Revenue Calculation"
    ],

    "Result": [
        original_records,
        cleaned_records,
        len(valid_order_records),
        round(total_order_value, 2),
        round(mean_order_value, 2),
        round(median_order_value, 2),
        payment_counts.idxmax() if not payment_counts.empty else "No Data",
        channel_counts.idxmax() if not channel_counts.empty else "No Data",
        weather_counts.idxmax() if not weather_counts.empty else "No Data",
        round(mean_delivery_time, 2),
        round(median_delivery_time, 2),
        len(valid_delivery_times),
        round(total_platform_revenue, 2),
        included_revenue_records
    ]
})

display(business_summary)

,Business Metric,Result
0,Original Dataset Records,1010
1,Cleaned Dataset Records,1000
2,Records with Valid Order Value,960
3,Total Order Value,1013415.0
4,Mean Order Value,1055.64
5,Median Order Value,1040.0
6,Most Used Payment Method,Cash on Delivery
7,Most Used Order Channel,Mobile App
8,Most Frequent Weather Condition,Sunny
9,Mean Delivery Time (Minutes),73.58


### Task 6 — Business Analysis Observations

After completing the business analysis, I understood how the food-delivery company's order data can be used to find useful business information.

1. **Dataset records:** The original dataset contained [original count] records, and [cleaned count] records remained after removing exact duplicates.

2. **Order Value:** Out of the cleaned records, [valid count] records had valid Order Values. I calculated the total, mean and median using these records only.

3. **Payment Method:** [Payment method] was the most frequently used payment method among orders with known payment information.

4. **Order Channel:** [Order channel] received the highest number of orders.

5. **Weather Condition:** [Weather condition] was the most frequently recorded weather condition.

6. **Delivery Performance:** The mean delivery time was [mean] minutes, and the median delivery time was [median] minutes. Only delivery durations between 0 and 120 minutes were included.

7. **Restaurant Analysis:** The top five restaurants were identified based on their order counts using Restaurant ID.

8. **Platform Net Revenue:** The total simplified Platform Net Revenue was [revenue amount]. This calculation included [number] records where all four required financial components were available.

**Conclusion:**

Through this analysis, I learned how Pandas can be used to calculate business metrics from cleaned data. These results help understand customer ordering patterns, popular payment methods, restaurant performance, delivery time and platform revenue.

I also understood why missing financial information should be handled carefully, because assuming missing values are zero can affect the accuracy of business calculations.

# Task 7 — Business Analysis Observations

After completing the business analysis, I reviewed the results obtained from the cleaned food orders dataset. The analysis helped me understand the order details, customer payment preferences, delivery performance and platform revenue.

### 7.1 Observations from the Analysis

1. **Dataset Records:** The original dataset contained [original count] records. After cleaning and removing exact duplicate rows, [cleaned count] records remained.

2. **Order Value:** Out of the cleaned records, [valid count] records had valid Order Values. I used these records to calculate the total, mean and median order values. This helped me understand the overall order value and the average amount spent per order.

3. **Payment Method:** [Payment method] was the most commonly used payment method among orders with known payment information. This shows which payment option was used more frequently by customers.

4. **Order Channel:** [Order channel] received the highest number of orders. From this result, I understood which platform customers used most often to place their food orders.

5. **Weather Condition:** [Weather condition] was the most frequently recorded weather condition in the dataset. This gives an idea of the weather conditions recorded during the orders.

6. **Delivery Performance:** The mean delivery time was [mean] minutes, while the median delivery time was [median] minutes. I considered only delivery durations between 0 and 120 minutes for this calculation.

7. **Restaurant Analysis:** I identified the five restaurants with the highest number of orders using Restaurant ID. This helped me compare restaurant order counts and identify the restaurants receiving more orders.

8. **Platform Net Revenue:** The total simplified Platform Net Revenue was [revenue amount]. A total of [number] records were included because all four required financial components were available for those records.

### 7.2 Conclusion

Through this assignment, I learned how to understand, clean, transform and analyse a food orders dataset using Python and Pandas.

I also learned how to use functions such as groupby(), value_counts(), sum(), mean() and median() to answer business-related questions.

The analysis helped me understand customer ordering patterns, commonly used payment methods, popular order channels, delivery performance and platform revenue.

I understood that data cleaning is important before performing analysis because missing values, duplicate records and incorrect data can affect the final results. I also learned that missing financial values should not automatically be replaced with zero, as this can lead to incorrect business calculations.

# Task 8 — Validate the Transformed Dataset

8.1 Check whether the cleaned dataset has the expected columns

The first step is to check whether the cleaned dataset contains all the required original and derived columns. This helps confirm that no important columns were accidentally removed during cleaning and transformation.

In [42]:
# Check the columns available in the cleaned dataset
print("Columns in the cleaned dataset:")
print(work_df.columns.tolist())

# Check the expected columns
expected_columns = [
    "Order ID",
    "Order Value",
    "Delivery Fee",
    "Commission Fee",
    "Payment Processing Fee",
    "Refunds/Chargebacks",
    "Payment Method",
    "Order Channel",
    "Weather Condition",
    "Delivery_Time_Minutes",
    "Order_Date",
    "Order_Hour",
    "Platform_Net_Revenue"
]

missing_columns = [
    col for col in expected_columns
    if col not in work_df.columns
]

if len(missing_columns) == 0:
    print("\nPASS: All expected columns are available.")
else:
    print("\nFAIL: Some expected columns are missing:")
    print(missing_columns)

Columns in the cleaned dataset:
['Order ID', 'Customer ID', 'Restaurant ID', 'Order Date and Time', 'Delivery Date and Time', 'Order Value', 'Delivery Fee', 'Payment Method', 'Discounts and Offers', 'Commission Fee', 'Payment Processing Fee', 'Refunds/Chargebacks', 'Weather Condition', 'Order Channel', 'Delivery_Time_Minutes', 'Delivery_Duration_Flag', 'Order_Date', 'Order_Hour', 'Platform_Net_Revenue']

PASS: All expected columns are available.


8.2 Check datetime and financial column data types

Datetime columns should have a datetime data type, while financial columns should have numeric data types. This is important because incorrect data types can cause errors during calculations and analysis.

In [43]:
# Display the data types of all columns
print("Data types in the cleaned dataset:")
print(work_df.dtypes)

# Check datetime columns
datetime_columns = [
    "Order Date",
    "Delivery Date",
    "Order_Date"
]

for col in datetime_columns:
    if col in work_df.columns:
        print(
            col,
            ":", work_df[col].dtype,
            "| Valid:",
            pd.api.types.is_datetime64_any_dtype(work_df[col])
        )

# Check financial columns
financial_columns = [
    "Order Value",
    "Delivery Fee",
    "Commission Fee",
    "Payment Processing Fee",
    "Refunds/Chargebacks",
    "Platform_Net_Revenue"
]

for col in financial_columns:
    if col in work_df.columns:
        print(
            col,
            ":", work_df[col].dtype,
            "| Numeric:",
            pd.api.types.is_numeric_dtype(work_df[col])
        )

Data types in the cleaned dataset:
Order ID                           int64
Customer ID                       object
Restaurant ID                     object
Order Date and Time       datetime64[ns]
Delivery Date and Time    datetime64[ns]
Order Value                      float64
Delivery Fee                     float64
Payment Method                    object
Discounts and Offers              object
Commission Fee                   float64
Payment Processing Fee           float64
Refunds/Chargebacks              float64
Weather Condition                 object
Order Channel                     object
Delivery_Time_Minutes            float64
Delivery_Duration_Flag            object
Order_Date                        object
Order_Hour                         int32
Platform_Net_Revenue             float64
dtype: object
Order_Date : object | Valid: False
Order Value : float64 | Numeric: True
Delivery Fee : float64 | Numeric: True
Commission Fee : float64 | Numeric: True
Payment Processing 

8.3 Check whether fully duplicated rows have been handled

Fully duplicated records were identified during the data quality assessment. According to the cleaning rules, exact duplicate rows were removed from the working dataset. This check confirms whether any fully duplicated rows remain.

In [44]:
# Count fully duplicated rows in the original dataset
original_duplicates = raw_df.duplicated().sum()

# Count fully duplicated rows in the cleaned dataset
cleaned_duplicates = work_df.duplicated().sum()

print("Fully duplicated rows in original dataset:",
      original_duplicates)

print("Fully duplicated rows in cleaned dataset:",
      cleaned_duplicates)

if cleaned_duplicates == 0:
    print("PASS: No fully duplicated rows remain.")
else:
    print("CHECK: Fully duplicated rows are still present.")

Fully duplicated rows in original dataset: 10
Fully duplicated rows in cleaned dataset: 0
PASS: No fully duplicated rows remain.


8.4 Check and explain repeated Order ID values

Repeated Order IDs were checked separately from fully duplicated rows. A repeated Order ID does not always mean that the entire record is duplicated. Therefore, these records were inspected and not automatically removed.

In [45]:
# Find repeated Order IDs
repeated_order_ids = work_df[
    work_df["Order ID"].duplicated(keep=False)
].sort_values("Order ID")

print("Number of records with repeated Order IDs:",
      len(repeated_order_ids))

print("\nRepeated Order ID records:")
display(repeated_order_ids)

if repeated_order_ids.empty:
    print("PASS: No repeated Order IDs found.")
else:
    print("CHECK: Repeated Order IDs were identified and retained for inspection.")

Number of records with repeated Order IDs: 0

Repeated Order ID records:


,Order ID,Customer ID,Restaurant ID,Order Date and Time,Delivery Date and Time,Order Value,Delivery Fee,Payment Method,Discounts and Offers,Commission Fee,Payment Processing Fee,Refunds/Chargebacks,Weather Condition,Order Channel,Delivery_Time_Minutes,Delivery_Duration_Flag,Order_Date,Order_Hour,Platform_Net_Revenue


PASS: No repeated Order IDs found.


8.5 Check negative delivery durations

Delivery duration should not be negative because a delivery cannot be completed before an order is placed. This check identifies negative delivery durations and verifies that no negative values remain among the records classified as valid. Any flagged records should be documented rather than silently ignored.

In [46]:
# Identify negative delivery durations
negative_delivery_records = work_df[
    work_df["Delivery_Time_Minutes"] < 0
]

# Identify valid delivery durations
# Use the same valid-duration criteria documented in Task 5.
valid_delivery_records = work_df[
    work_df["Delivery_Time_Minutes"].notna()
    & (work_df["Delivery_Time_Minutes"] >= 0)
]

print("Negative delivery duration records:",
      len(negative_delivery_records))

print("Records classified as valid:",
      len(valid_delivery_records))

print("\nNegative duration records:")
display(negative_delivery_records)

# Validate the records classified as valid
if (valid_delivery_records["Delivery_Time_Minutes"] < 0).any():
    print("FAIL: Negative delivery durations exist in valid records.")
else:
    print("PASS: No negative delivery durations exist in valid records.")

Negative delivery duration records: 0
Records classified as valid: 1000

Negative duration records:


,Order ID,Customer ID,Restaurant ID,Order Date and Time,Delivery Date and Time,Order Value,Delivery Fee,Payment Method,Discounts and Offers,Commission Fee,Payment Processing Fee,Refunds/Chargebacks,Weather Condition,Order Channel,Delivery_Time_Minutes,Delivery_Duration_Flag,Order_Date,Order_Hour,Platform_Net_Revenue


PASS: No negative delivery durations exist in valid records.


8.6 Verify delivery-time calculations manually

A few records are selected to verify whether the calculated delivery duration matches the difference between the delivery time and order time. This helps confirm that the derived column was calculated correctly.

In [49]:
print(work_df.columns.tolist())
print(work_df.dtypes)

['Order ID', 'Customer ID', 'Restaurant ID', 'Order Date and Time', 'Delivery Date and Time', 'Order Value', 'Delivery Fee', 'Payment Method', 'Discounts and Offers', 'Commission Fee', 'Payment Processing Fee', 'Refunds/Chargebacks', 'Weather Condition', 'Order Channel', 'Delivery_Time_Minutes', 'Delivery_Duration_Flag', 'Order_Date', 'Order_Hour', 'Platform_Net_Revenue']
Order ID                           int64
Customer ID                       object
Restaurant ID                     object
Order Date and Time       datetime64[ns]
Delivery Date and Time    datetime64[ns]
Order Value                      float64
Delivery Fee                     float64
Payment Method                    object
Discounts and Offers              object
Commission Fee                   float64
Payment Processing Fee           float64
Refunds/Chargebacks              float64
Weather Condition                 object
Order Channel                     object
Delivery_Time_Minutes            float64
Delivery_D

In [50]:
# Select 5 records where both timestamps are available
sample_records = work_df.dropna(
    subset=[
        "Order Date and Time",
        "Delivery Date and Time",
        "Delivery_Time_Minutes"
    ]
).head(5).copy()

# Recalculate delivery duration manually
sample_records["Recalculated_Delivery_Time"] = (
    sample_records["Delivery Date and Time"]
    - sample_records["Order Date and Time"]
).dt.total_seconds() / 60

# Compare the original and recalculated delivery times
sample_records["Calculation_Matches"] = (
    sample_records["Delivery_Time_Minutes"].round(2)
    == sample_records["Recalculated_Delivery_Time"].round(2)
)

# Display the verification results
display(
    sample_records[
        [
            "Order Date and Time",
            "Delivery Date and Time",
            "Delivery_Time_Minutes",
            "Recalculated_Delivery_Time",
            "Calculation_Matches"
        ]
    ]
)

,Order Date and Time,Delivery Date and Time,Delivery_Time_Minutes,Recalculated_Delivery_Time,Calculation_Matches
0,2024-02-01 01:11:52,2024-02-01 02:39:52,88.0,88.0,True
1,2024-02-02 22:11:04,2024-02-02 22:46:04,35.0,35.0,True
2,2024-01-31 05:54:35,2024-01-31 06:52:35,58.0,58.0,True
3,2024-01-16 22:52:49,2024-01-16 23:38:49,46.0,46.0,True
4,2024-01-29 01:19:30,2024-01-29 02:48:30,89.0,89.0,True


Task 8.7 — Verify Platform Net Revenue Calculation.

In [51]:
# Select 5 records where all revenue components are available
revenue_check = work_df.dropna(
    subset=[
        "Commission Fee",
        "Delivery Fee",
        "Payment Processing Fee",
        "Refunds/Chargebacks",
        "Platform_Net_Revenue"
    ]
).head(5).copy()

# Recalculate platform net revenue manually
revenue_check["Recalculated_Revenue"] = (
    revenue_check["Commission Fee"]
    + revenue_check["Delivery Fee"]
    - revenue_check["Payment Processing Fee"]
    - revenue_check["Refunds/Chargebacks"]
)

# Compare calculated and stored revenue
revenue_check["Calculation_Matches"] = (
    revenue_check["Platform_Net_Revenue"].round(2)
    == revenue_check["Recalculated_Revenue"].round(2)
)

# Display verification results
display(
    revenue_check[
        [
            "Commission Fee",
            "Delivery Fee",
            "Payment Processing Fee",
            "Refunds/Chargebacks",
            "Platform_Net_Revenue",
            "Recalculated_Revenue",
            "Calculation_Matches"
        ]
    ]
)

,Commission Fee,Delivery Fee,Payment Processing Fee,Refunds/Chargebacks,Platform_Net_Revenue,Recalculated_Revenue,Calculation_Matches
0,150.0,0.0,47.0,0.0,103.0,103.0,True
1,198.0,40.0,23.0,0.0,215.0,215.0,True
2,195.0,30.0,45.0,0.0,180.0,180.0,True
5,92.0,20.0,27.0,150.0,-65.0,-65.0,True
6,144.0,30.0,12.0,50.0,112.0,112.0,True


**Revenue Calculation Verification**

I selected five records with all the required financial values available and recalculated Platform Net Revenue using the formula:

Platform Net Revenue = Commission Fee + Delivery Fee − Payment Processing Fee − Refunds/Chargebacks.

I compared the recalculated values with the existing `Platform_Net_Revenue` column. The `Calculation_Matches` column shows whether both values are equal. This helps verify that the revenue calculation was performed correctly.

Task 8.8 — Compare Missing Values Before and After Cleaning

In [52]:
# Get the original columns available in both datasets
original_columns = raw_df.columns.intersection(work_df.columns)

# Calculate missing values before and after cleaning
missing_comparison = pd.DataFrame({
    "Missing Before Cleaning": raw_df[original_columns].isnull().sum(),
    "Missing After Cleaning": work_df[original_columns].isnull().sum()
})

# Calculate missing percentages
missing_comparison["Before Missing %"] = (
    missing_comparison["Missing Before Cleaning"] / len(raw_df) * 100
).round(2)

missing_comparison["After Missing %"] = (
    missing_comparison["Missing After Cleaning"] / len(work_df) * 100
).round(2)

display(missing_comparison)

,Missing Before Cleaning,Missing After Cleaning,Before Missing %,After Missing %
Order ID,0,0,0.00,0.0
Customer ID,0,0,0.00,0.0
Restaurant ID,0,0,0.00,0.0
Order Date and Time,0,0,0.00,0.0
Delivery Date and Time,0,0,0.00,0.0
Order Value,41,40,4.06,4.0
Delivery Fee,50,50,4.95,5.0
Payment Method,50,0,4.95,0.0
Discounts and Offers,242,0,23.96,0.0
Commission Fee,40,40,3.96,4.0


**Missing Values Comparison**

I compared the missing values in the original dataset with those in the cleaned dataset. This helped me understand which missing values were handled during cleaning and which were retained.

Some missing values were intentionally kept because replacing them with zero or another value could affect the accuracy of the financial analysis.

Task 8.9 — Verify Records Removed, Retained and Excluded

In [53]:
# Record count comparison
original_count = len(raw_df)
cleaned_count = len(work_df)
removed_count = original_count - cleaned_count

# Records with available, non-negative delivery duration
valid_delivery_mask = (
    work_df["Delivery_Time_Minutes"].notna()
    & (work_df["Delivery_Time_Minutes"] >= 0)
)

valid_delivery_count = valid_delivery_mask.sum()
excluded_delivery_count = len(work_df) - valid_delivery_count

# Records with all required revenue values available
revenue_columns = [
    "Commission Fee",
    "Delivery Fee",
    "Payment Processing Fee",
    "Refunds/Chargebacks"
]

valid_revenue_mask = work_df[revenue_columns].notna().all(axis=1)

valid_revenue_count = valid_revenue_mask.sum()
excluded_revenue_count = len(work_df) - valid_revenue_count

# Display record summary
validation_summary = pd.DataFrame({
    "Category": [
        "Original Records",
        "Cleaned Records",
        "Records Removed",
        "Records with Available Non-negative Delivery Duration",
        "Records Excluded from Delivery Duration Calculation",
        "Records with All Revenue Components Available",
        "Records Excluded from Revenue Calculation"
    ],
    "Count": [
        original_count,
        cleaned_count,
        removed_count,
        valid_delivery_count,
        excluded_delivery_count,
        valid_revenue_count,
        excluded_revenue_count
    ]
})

display(validation_summary)

,Category,Count
0,Original Records,1010
1,Cleaned Records,1000
2,Records Removed,10
3,Records with Available Non-negative Delivery D...,1000
4,Records Excluded from Delivery Duration Calcul...,0
5,Records with All Revenue Components Available,828
6,Records Excluded from Revenue Calculation,172


**Record Count Verification**

I compared the number of records in the original and cleaned datasets. I also checked how many records were available for delivery-time and revenue calculations.

Records with missing or invalid delivery durations were excluded from the delivery-time calculation. Records with missing financial components were excluded from the platform net revenue calculation because all required values are needed for the formula.

Task 8.10 — Final Data Validation Summary

### Final Validation Summary

After completing the data cleaning and transformation steps, I performed validation checks to verify the quality of the cleaned dataset.

The validation included:

1. Checking whether all expected columns were available.
2. Verifying the data types of columns.
3. Checking for duplicate records and repeated Order IDs.
4. Identifying negative or invalid delivery durations.
5. Manually verifying delivery-time calculations using order and delivery timestamps.
6. Checking the Platform Net Revenue formula using the required financial columns.
7. Comparing missing values before and after cleaning.
8. Comparing the original and cleaned record counts and identifying records excluded from calculations.

These checks helped verify that the cleaning and transformation steps were applied correctly. The original dataset was preserved, and the cleaned dataset was used for further analysis.